# FFmpeg Processing Plugin

> FFmpeg-based media processing plugin implementing the `MediaProcessingPlugin` interface.

In [ ]:
#| default_exp plugin

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
import json
import logging
import os
import shutil
import subprocess
import time
import uuid
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Union

from cjm_media_plugin_system.processing_interface import MediaProcessingPlugin
from cjm_media_plugin_system.core import MediaMetadata
from cjm_media_plugin_system.storage import MediaProcessingStorage

from cjm_plugin_system.utils.hashing import hash_file
from cjm_plugin_system.utils.validation import (
    dict_to_config, config_to_dict, dataclass_to_jsonschema,
    SCHEMA_TITLE, SCHEMA_DESC, SCHEMA_ENUM
)

from cjm_ffmpeg_utils.core import FFMPEG_AVAILABLE, get_audio_codec
from cjm_ffmpeg_utils.media_info import get_audio_info_ffmpeg, get_media_duration
from cjm_ffmpeg_utils.audio import extract_audio_segment, convert_to_mp3
from cjm_ffmpeg_utils.execution import run_ffmpeg_with_progress

from cjm_media_plugin_ffmpeg.meta import get_plugin_metadata

## Configuration

In [ ]:
#| export
@dataclass
class FFmpegPluginConfig:
    """Configuration for the FFmpeg processing plugin."""

    output_dir: Optional[str] = field(
        default=None,
        metadata={
            SCHEMA_TITLE: "Output Directory",
            SCHEMA_DESC: "Default output directory for processed files. Uses plugin data_dir if None."
        }
    )

    default_audio_format: str = field(
        default="mp3",
        metadata={
            SCHEMA_TITLE: "Default Audio Format",
            SCHEMA_DESC: "Default format for audio extraction and conversion.",
            SCHEMA_ENUM: ["mp3", "wav", "flac", "aac", "ogg", "m4a"]
        }
    )

    default_audio_bitrate: str = field(
        default="192k",
        metadata={
            SCHEMA_TITLE: "Default Audio Bitrate",
            SCHEMA_DESC: "Default bitrate for audio encoding operations.",
            SCHEMA_ENUM: ["128k", "192k", "256k", "320k"]
        }
    )

    prefer_stream_copy: bool = field(
        default=True,
        metadata={
            SCHEMA_TITLE: "Prefer Stream Copy",
            SCHEMA_DESC: "When possible, copy audio stream without re-encoding (faster, lossless)."
        }
    )

In [ ]:
schema = dataclass_to_jsonschema(FFmpegPluginConfig)
assert "properties" in schema
assert "default_audio_format" in schema["properties"]
assert schema["properties"]["default_audio_format"]["default"] == "mp3"
print(f"Config schema: {list(schema['properties'].keys())}")

Config schema: ['output_dir', 'default_audio_format', 'default_audio_bitrate', 'prefer_stream_copy']


## Plugin Class

In [ ]:
#| export
class FFmpegProcessingPlugin(MediaProcessingPlugin):
    """FFmpeg-based media processing plugin."""

    config_class = FFmpegPluginConfig

    def __init__(self):
        """Initialize the FFmpeg processing plugin."""
        self.logger = logging.getLogger(f"{__name__}.{type(self).__name__}")
        self.config: Optional[FFmpegPluginConfig] = None
        self.storage: Optional[MediaProcessingStorage] = None
        self._data_dir: Optional[str] = None

    @property
    def name(self) -> str:  # Plugin identifier
        return "ffmpeg"

    @property
    def version(self) -> str:  # Plugin version
        return "1.0.0"

    @property
    def supported_media_types(self) -> List[str]:  # Supported input types
        return ["audio", "video"]

    def initialize(self, config: Optional[Any] = None) -> None:
        """Initialize plugin with configuration."""
        self.config = dict_to_config(FFmpegPluginConfig, config or {})
        meta = get_plugin_metadata()
        db_path = meta["db_path"]
        self._data_dir = os.path.dirname(db_path)
        self.storage = MediaProcessingStorage(db_path)
        self.logger.info(f"Initialized FFmpeg plugin (format={self.config.default_audio_format})")

    def get_config_schema(self) -> Dict[str, Any]:  # JSON Schema for UI form generation
        """Return the JSON Schema for plugin configuration."""
        return dataclass_to_jsonschema(FFmpegPluginConfig)

    def get_current_config(self) -> Dict[str, Any]:  # Current config as dict
        """Return the current configuration as a dictionary."""
        return config_to_dict(self.config) if self.config else {}

    def is_available(self) -> bool:  # Whether ffmpeg is installed
        """Check if ffmpeg is available on this system."""
        return FFMPEG_AVAILABLE

    def cleanup(self) -> None:
        """Clean up plugin resources."""
        self.logger.info("FFmpeg plugin cleaned up")

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------

    def _get_output_dir(self,
                        output_dir: Optional[str] = None,  # Explicit output dir override
                        subdirectory: Optional[str] = None,  # Subdirectory within output dir
                       ) -> str:  # Resolved output directory path
        """Resolve the output directory, creating it if needed."""
        base = output_dir or self.config.output_dir or self._data_dir
        if subdirectory:
            base = os.path.join(base, subdirectory)
        os.makedirs(base, exist_ok=True)
        return base

    def _store_job(self,
                   action: str,  # Action name
                   input_path: str,  # Source file path
                   output_path: str,  # Output file path
                   parameters: Optional[Dict[str, Any]] = None,  # Action parameters
                   metadata: Optional[Dict[str, Any]] = None,  # Additional metadata
                  ) -> str:  # Generated job_id
        """Hash input/output files and store a processing job record."""
        job_id = str(uuid.uuid4())
        input_hash = hash_file(input_path)
        output_hash = hash_file(output_path)
        self.storage.save(
            job_id=job_id, action=action,
            input_path=input_path, input_hash=input_hash,
            output_path=output_path, output_hash=output_hash,
            parameters=parameters, metadata=metadata
        )
        return job_id

    def _detect_audio_codec(self,
                            file_path: str,  # Path to media file
                           ) -> Optional[str]:  # Audio codec name or None
        """Detect the audio codec in a media file via ffprobe."""
        cmd = [
            'ffprobe', '-v', 'quiet',
            '-select_streams', 'a:0',
            '-show_entries', 'stream=codec_name',
            '-of', 'csv=p=0',
            file_path
        ]
        try:
            result = subprocess.run(cmd, check=True, capture_output=True, text=True)
            codec = result.stdout.strip()
            return codec if codec else None
        except subprocess.CalledProcessError:
            return None

    # ------------------------------------------------------------------
    # Action dispatch
    # ------------------------------------------------------------------

    def execute(self,
                action: str = "get_info",  # Action to perform
                **kwargs
               ) -> Dict[str, Any]:  # Action result
        """Dispatch to the appropriate action handler."""
        if action == "get_info":
            return self.get_info(kwargs["file_path"]).to_dict()
        elif action == "convert":
            output = self.convert(kwargs["input_path"], kwargs["output_format"], **{
                k: v for k, v in kwargs.items() if k not in ("input_path", "output_format")
            })
            return {"output_path": output}
        elif action == "extract_segment":
            output = self.extract_segment(
                kwargs["input_path"], kwargs["start"], kwargs["end"],
                kwargs.get("output_path")
            )
            return {"output_path": output}
        elif action == "extract_audio":
            return self._extract_audio(**kwargs)
        elif action == "segment_audio":
            return self._segment_audio(**kwargs)
        else:
            raise ValueError(f"Unknown action: {action}")

    # ------------------------------------------------------------------
    # Core actions
    # ------------------------------------------------------------------

    def get_info(self,
                 file_path: Union[str, Path],  # Path to media file
                ) -> MediaMetadata:  # Probed metadata
        """Get metadata for a media file via ffprobe."""
        file_path = str(file_path)
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File not found: {file_path}")

        cmd = [
            'ffprobe', '-v', 'quiet',
            '-show_format', '-show_streams', '-of', 'json',
            file_path
        ]
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        probe_data = json.loads(result.stdout)

        format_info = probe_data.get('format', {})
        duration = float(format_info.get('duration', 0))
        size_bytes = int(format_info.get('size', 0))
        container_format = format_info.get('format_name', '')

        video_streams = []
        audio_streams = []
        for stream in probe_data.get('streams', []):
            codec_type = stream.get('codec_type', '')
            if codec_type == 'video':
                video_streams.append({
                    'codec': stream.get('codec_name'),
                    'width': stream.get('width'),
                    'height': stream.get('height'),
                    'fps': stream.get('r_frame_rate'),
                })
            elif codec_type == 'audio':
                stream_duration = float(stream.get('duration', duration))
                audio_streams.append({
                    'codec': stream.get('codec_name'),
                    'sample_rate': int(stream.get('sample_rate', 0)),
                    'channels': int(stream.get('channels', 0)),
                    'duration': stream_duration,
                })

        return MediaMetadata(
            path=file_path,
            duration=duration,
            format=container_format,
            size_bytes=size_bytes,
            video_streams=video_streams,
            audio_streams=audio_streams,
        )

    def convert(self,
                input_path: Union[str, Path],  # Source file path
                output_format: str,  # Target format (e.g. 'mp3', 'wav')
                **kwargs
               ) -> str:  # Output file path
        """Convert media to a different format."""
        input_path = str(input_path)
        if not os.path.exists(input_path):
            raise FileNotFoundError(f"File not found: {input_path}")

        bitrate = kwargs.get('bitrate', self.config.default_audio_bitrate)
        sample_rate = kwargs.get('sample_rate')
        channels = kwargs.get('channels')

        stem = Path(input_path).stem
        out_dir = self._get_output_dir(subdirectory="converted")
        output_path = os.path.join(out_dir, f"{stem}.{output_format}")

        total_duration = get_media_duration(Path(input_path))
        codec = get_audio_codec(output_format)

        cmd = ['ffmpeg', '-i', input_path, '-vn']
        cmd.extend(['-acodec', codec])
        if bitrate:
            cmd.extend(['-b:a', bitrate])
        if sample_rate:
            cmd.extend(['-ar', str(sample_rate)])
        if channels:
            cmd.extend(['-ac', str(channels)])
        cmd.extend(['-progress', 'pipe:2', '-y', output_path])

        self.report_progress(0.0, f"Converting to {output_format}...")
        run_ffmpeg_with_progress(
            cmd=cmd, total_duration=total_duration,
            description=f"Converting to {output_format}"
        )

        job_id = self._store_job(
            action="convert", input_path=input_path, output_path=output_path,
            parameters={"output_format": output_format, "bitrate": bitrate,
                         "sample_rate": sample_rate, "channels": channels},
            metadata={"duration": total_duration}
        )
        self.report_progress(1.0, "Complete")
        self.logger.info(f"Converted {input_path} -> {output_path} (job={job_id})")
        return output_path

    def extract_segment(self,
                        input_path: Union[str, Path],  # Source audio file
                        start: float,  # Start time in seconds
                        end: float,  # End time in seconds
                        output_path: Optional[str] = None,  # Custom output path
                       ) -> str:  # Output file path
        """Extract a temporal segment from a media file."""
        input_path = str(input_path)
        if not os.path.exists(input_path):
            raise FileNotFoundError(f"File not found: {input_path}")

        duration = end - start
        if duration <= 0:
            raise ValueError(f"end ({end}) must be greater than start ({start})")

        if output_path is None:
            ext = Path(input_path).suffix
            stem = Path(input_path).stem
            out_dir = self._get_output_dir(subdirectory="segments")
            output_path = os.path.join(out_dir, f"{stem}_{start:.2f}_{end:.2f}{ext}")

        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        extract_audio_segment(
            input_path=Path(input_path),
            output_path=Path(output_path),
            start_time=str(start),
            duration=str(duration),
        )

        job_id = self._store_job(
            action="extract_segment", input_path=input_path, output_path=output_path,
            parameters={"start": start, "end": end, "duration": duration},
        )
        self.logger.info(f"Extracted segment [{start:.2f}-{end:.2f}] -> {output_path} (job={job_id})")
        return output_path

    # ------------------------------------------------------------------
    # Custom actions
    # ------------------------------------------------------------------

    def _extract_audio(self,
                       input_path: str,  # Path to video file
                       output_format: Optional[str] = None,  # Audio format (default: from codec detection)
                       output_dir: Optional[str] = None,  # Output directory override
                      ) -> Dict[str, Any]:  # Extraction result with output_path, job_id, duration, codec
        """Extract audio stream from a video file."""
        input_path = str(input_path)
        if not os.path.exists(input_path):
            raise FileNotFoundError(f"File not found: {input_path}")

        self.report_progress(0.0, "Probing video file...")

        # Detect audio codec
        codec = self._detect_audio_codec(input_path)
        if not codec:
            raise ValueError(f"No audio stream found in: {input_path}")

        # Determine output format and whether to stream copy
        stream_copy = self.config.prefer_stream_copy
        if output_format is None:
            # Map codec to file extension for stream copy
            codec_ext_map = {
                'aac': 'm4a', 'mp3': 'mp3', 'vorbis': 'ogg', 'opus': 'ogg',
                'flac': 'flac', 'pcm_s16le': 'wav', 'pcm_s24le': 'wav',
            }
            output_format = codec_ext_map.get(codec, self.config.default_audio_format)
            if codec not in codec_ext_map:
                stream_copy = False
        else:
            # User specified format — check if codec matches
            expected_codec = get_audio_codec(output_format)
            if expected_codec != 'copy' and expected_codec != codec:
                stream_copy = False

        stem = Path(input_path).stem
        out_dir = self._get_output_dir(output_dir, subdirectory="extracted")
        output_path = os.path.join(out_dir, f"{stem}.{output_format}")

        total_duration = get_media_duration(Path(input_path))

        # Build ffmpeg command
        cmd = ['ffmpeg', '-i', input_path, '-vn']
        if stream_copy:
            cmd.extend(['-acodec', 'copy'])
        else:
            target_codec = get_audio_codec(output_format)
            cmd.extend(['-acodec', target_codec])
            cmd.extend(['-b:a', self.config.default_audio_bitrate])
        cmd.extend(['-progress', 'pipe:2', '-y', output_path])

        self.report_progress(0.1, "Extracting audio stream...")
        run_ffmpeg_with_progress(
            cmd=cmd, total_duration=total_duration,
            description="Extracting audio"
        )

        self.report_progress(0.9, "Hashing output file...")
        job_id = self._store_job(
            action="extract_audio", input_path=input_path, output_path=output_path,
            parameters={"output_format": output_format, "stream_copy": stream_copy},
            metadata={"codec": codec, "duration": total_duration,
                       "source_video_path": input_path}
        )

        self.report_progress(1.0, "Complete")
        self.logger.info(f"Extracted audio {input_path} -> {output_path} (stream_copy={stream_copy}, job={job_id})")

        return {
            "job_id": job_id,
            "output_path": output_path,
            "input_path": input_path,
            "duration": total_duration,
            "codec": codec,
            "stream_copy": stream_copy,
        }

    def _segment_audio(self,
                       input_path: str,  # Source audio file
                       boundaries: List[Dict[str, float]],  # List of {"start": float, "end": float}
                       output_dir: Optional[str] = None,  # Output directory override
                       output_format: Optional[str] = None,  # Output format (default: same as input)
                       filename_template: str = "segment_{index:03d}",  # Filename pattern
                      ) -> Dict[str, Any]:  # Segmentation result with segments list
        """Split audio file into segments at specified boundaries."""
        input_path = str(input_path)
        if not os.path.exists(input_path):
            raise FileNotFoundError(f"File not found: {input_path}")

        if not boundaries:
            raise ValueError("boundaries list must not be empty")

        # Validate boundaries are sorted and non-overlapping
        for i, b in enumerate(boundaries):
            if b["end"] <= b["start"]:
                raise ValueError(f"Boundary {i}: end ({b['end']}) must be > start ({b['start']})")
            if i > 0 and b["start"] < boundaries[i - 1]["end"]:
                raise ValueError(
                    f"Boundary {i} start ({b['start']}) overlaps with boundary {i-1} end ({boundaries[i-1]['end']})"
                )

        ext = output_format or Path(input_path).suffix.lstrip('.')
        stem = Path(input_path).stem
        out_dir = self._get_output_dir(output_dir, subdirectory=f"segments/{stem}")
        batch_key = str(uuid.uuid4())
        total = len(boundaries)

        self.report_progress(0.0, f"Segmenting into {total} segments...")

        segments = []
        for i, boundary in enumerate(boundaries):
            start = boundary["start"]
            end = boundary["end"]
            duration = end - start

            filename = f"{filename_template.format(index=i)}.{ext}"
            output_path = os.path.join(out_dir, filename)

            extract_audio_segment(
                input_path=Path(input_path),
                output_path=Path(output_path),
                start_time=str(start),
                duration=str(duration),
            )

            job_id = self._store_job(
                action="segment_audio", input_path=input_path, output_path=output_path,
                parameters={"start": start, "end": end, "index": i, "batch_key": batch_key},
                metadata={"source_audio": input_path, "segment_count": total,
                           "filename_template": filename_template}
            )

            segments.append({
                "job_id": job_id,
                "index": i,
                "output_path": output_path,
                "start": start,
                "end": end,
                "duration": duration,
            })

            self.report_progress((i + 1) / total, f"Segment {i + 1}/{total}")

        total_duration = sum(s["duration"] for s in segments)
        self.report_progress(1.0, f"Complete: {total} segments")
        self.logger.info(f"Segmented {input_path} into {total} segments (batch_key={batch_key})")

        return {
            "segments": segments,
            "input_path": input_path,
            "segment_count": total,
            "total_duration": total_duration,
            "batch_key": batch_key,
        }

In [ ]:
plugin = FFmpegProcessingPlugin()
assert plugin.name == "ffmpeg"
assert plugin.version == "1.0.0"
assert plugin.supported_media_types == ["audio", "video"]
assert plugin.is_available() == FFMPEG_AVAILABLE
print(f"Plugin: {plugin.name} v{plugin.version}")
print(f"FFmpeg available: {plugin.is_available()}")

Plugin: ffmpeg v1.0.0
FFmpeg available: True


In [ ]:
plugin.initialize({})
assert plugin.config is not None
assert plugin.config.default_audio_format == "mp3"
assert plugin.config.prefer_stream_copy is True
assert plugin.storage is not None
print(f"Config: format={plugin.config.default_audio_format}, bitrate={plugin.config.default_audio_bitrate}")
print(f"Current config: {plugin.get_current_config()}")

Config: format=mp3, bitrate=192k
Current config: {'output_dir': None, 'default_audio_format': 'mp3', 'default_audio_bitrate': '192k', 'prefer_stream_copy': True}


In [ ]:
plugin.initialize({"default_audio_format": "wav", "default_audio_bitrate": "320k"})
assert plugin.config.default_audio_format == "wav"
assert plugin.config.default_audio_bitrate == "320k"
print(f"Custom config: {plugin.get_current_config()}")

Custom config: {'output_dir': None, 'default_audio_format': 'wav', 'default_audio_bitrate': '320k', 'prefer_stream_copy': True}


In [ ]:
schema = plugin.get_config_schema()
assert "properties" in schema
props = schema["properties"]
assert "output_dir" in props
assert "default_audio_format" in props
assert "default_audio_bitrate" in props
assert "prefer_stream_copy" in props
print(f"Schema properties: {list(props.keys())}")

Schema properties: ['output_dir', 'default_audio_format', 'default_audio_bitrate', 'prefer_stream_copy']


In [ ]:
plugin.cleanup()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()